# AI工学101 — 第39回

## ハイパーパラメータ探索：GridSearchCV → RandomizedSearchCV

よし、レベル。今日は **「勘でパラメータをいじる機械学習」から一段進む回** だ。💪🔥

前回までのモデル比較では、

```text
Linear
Random Forest
Gradient Boosting
```

を同じ条件で比較した。

でも実際には、Random Forestひとつ取っても、

```text
n_estimators = 100
max_depth = 3
```

と、

```text
n_estimators = 500
max_depth = None
```

では**別物に近い振る舞い**になる。

そこで今日やるのが、

> **ハイパーパラメータ探索を、再現可能な実験として設計する**

こと。

---

# 🎯 今日のゴール

* 学習パラメータとハイパーパラメータを区別する
* 探索空間を設計する
* `GridSearchCV` を使える
* `RandomizedSearchCV` を使える
* Pipeline内部のパラメータを探索できる
* CV結果を読み取れる
* `best_params_` と `best_estimator_` を使える
* 探索しすぎの危険を理解する
* Nested CVの役割を知る

---

# 📖 講義：約20分

## 1. 学習パラメータ vs ハイパーパラメータ

例えばLinear Regressionでは、

```text
y = wx + b
```

の、

```text
w
b
```

はデータからモデルが学習する。

これが、

> **学習パラメータ**

一方、

```python
Ridge(alpha=1.0)
```

の、

```text
alpha
```

は人間側が指定する。

これが、

> **ハイパーパラメータ**

---

## 2. 重要：探索空間も設計対象

例えば、

```python
max_depth = [2, 3, 5, 10, None]
```

とするか、

```python
max_depth = [2, 4, 8, 16]
```

とするか。

これは単なる設定ではない。

> **「この問題では、どの程度の複雑さのモデルが有力だろうか？」**

という仮説を、探索空間に埋め込んでいる。

つまり、

```text
探索空間
=
仮説空間の人間側の設計
```

だ。

ここ、認知科学の実験設計とかなり似ているぞ。

---

# 💻 実習1：まず手動で試す

前回の `RandomForestClassifier` を使う。

```python
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()

X = data.data
y = data.target
```

CV。

```python
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
```

まず手動。

```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

for depth in [3, 5, 10, None]:

    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=depth,
        random_state=42,
        n_jobs=-1
    )

    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="roc_auc"
    )

    print(
        depth,
        scores.mean()
    )
```

これは小規模なら有効。

でも、

```text
max_depth
min_samples_split
min_samples_leaf
n_estimators
max_features
```

と増えると、

人間が手作業で管理するのは面倒になる。

そこで。

---

# 🧠 3. Grid Search

`GridSearchCV` は、

> **指定した組み合わせを総当たりする**

。

例えば、

```python
param_grid = {
    "max_depth": [
        3,
        5,
        10
    ],

    "n_estimators": [
        100,
        300
    ]
}
```

なら、

```text
3 × 2 = 6通り
```

。

---

# 💻 実習2：GridSearchCV

```python
from sklearn.model_selection import GridSearchCV
```

。

モデル。

```python
rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)
```

探索空間。

```python
param_grid = {
    "n_estimators": [
        100,
        300
    ],

    "max_depth": [
        3,
        5,
        10,
        None
    ],

    "min_samples_split": [
        2,
        5,
        10
    ]
}
```

組み合わせ数。

```text
2 × 4 × 3
=
24通り
```

5-fold CVなら。

```text
24 × 5
=
120回の学習
```

になる。

ここ大事。

> **探索コストは組み合わせ数 × CV回数。**

---

実行。

```python
grid_search = GridSearchCV(
    estimator=rf,

    param_grid=param_grid,

    scoring="roc_auc",

    cv=cv,

    n_jobs=-1,

    return_train_score=True
)
```

学習。

```python
grid_search.fit(
    X,
    y
)
```

---

# 💻 実習3：結果を見る

最良パラメータ。

```python
print(
    grid_search.best_params_
)
```

最良スコア。

```python
print(
    grid_search.best_score_
)
```

最良モデル。

```python
best_model = grid_search.best_estimator_

print(
    best_model
)
```

---

# 🧠 4. `best_estimator_` は何か？

これは、

> **最良パラメータを使って、探索用データ全体で再学習されたモデル**

。

つまり、

```text
CVで比較
↓
最良パラメータ決定
↓
その設定でデータ全体にfit
```

。

ここは地味だけど便利。

---

# 💻 実習4：CV結果をDataFrame化

```python
import pandas as pd
```

。

```python
results = pd.DataFrame(
    grid_search.cv_results_
)
```

。

必要な列を見る。

```python
print(
    results[
        [
            "params",
            "mean_test_score",
            "std_test_score",
            "rank_test_score"
        ]
    ]
    .sort_values(
        "rank_test_score"
    )
    .head(10)
)
```

。

ここで、

```text
mean_test_score
```

だけではなく、

```text
std_test_score
```

も見る。

---

# 🧠 5. 最高平均 = 常に採用ではない

例えば。

```text
Model A

mean = 0.950
std  = 0.002
```

。

```text
Model B

mean = 0.952
std  = 0.020
```

。

Bの平均は少し高い。

でも分割によって性能がかなり変動する。

さらに、

```text
Model A
max_depth = 5
```

。

```text
Model B
max_depth = None
```

。

なら、

> **性能差が小さいなら単純なAを採用する**

という判断も合理的。

これは、

```text
Occam's Razor
```

的な発想。

---

# 🧠 6. Pipelineのパラメータ探索

ここから重要。

例えば、

```python
Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])
```

。

`C` を探索したい。

```python
param_grid = {
    "model__C": [
        0.01,
        0.1,
        1,
        10,
        100
    ]
}
```

。

この、

```text
model__C
```

がポイント。

```text
ステップ名
__
パラメータ名
```

。

---

# 💻 実習5：Pipeline + GridSearchCV

```python
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
```

。

```python
pipeline = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),

    (
        "model",
        LogisticRegression(
            max_iter=1000
        )
    )
])
```

。

探索。

```python
param_grid = {
    "model__C": [
        0.01,
        0.1,
        1,
        10,
        100
    ]
}
```

。

```python
search = GridSearchCV(
    estimator=pipeline,

    param_grid=param_grid,

    scoring="roc_auc",

    cv=cv,

    n_jobs=-1
)
```

。

```python
search.fit(
    X,
    y
)
```

。

```python
print(
    search.best_params_
)
```

。

---

# 🧠 7. なぜPipelineの中で探索するのか？

もし、

```text
StandardScaler
```

を全データに先にfitして、

その後でCVしたら。

```text
Validation fold
```

の情報が、

```text
scaler
```

に混ざる。

つまり、

> **Data Leakage**

。

Pipelineの中に入れると、

各foldで、

```text
Train fold
↓
Scaler fit

Validation fold
↓
Scaler transform only
```

になる。

だから、

> **前処理も含めてモデルとして評価する**

。

---

# 🎲 8. Grid Searchの問題

パラメータが増えると。

例えば、

```text
10個のパラメータ
```

。

それぞれ、

```text
5通り
```

。

組み合わせは、

```text
5¹⁰
=
9,765,625
```

。

5-fold CVなら。

```text
48,828,125回
```

の学習。

モデル「無理です」

GPU「知らんがな」

PC「死」

wwwww

そこで。

---

# 🧠 9. RandomizedSearchCV

```python
RandomizedSearchCV
```

。

これは、

> **探索空間からランダムに候補をサンプリングする**

。

全探索ではない。

でも、

> 大きな探索空間では、Grid Searchより効率的なことが多い。

---

# 💻 実習6：RandomizedSearchCV

```python
from sklearn.model_selection import RandomizedSearchCV
```

。

```python
param_distributions = {
    "n_estimators": [
        100,
        300,
        500,
        1000
    ],

    "max_depth": [
        3,
        5,
        10,
        20,
        None
    ],

    "min_samples_split": [
        2,
        5,
        10,
        20
    ],

    "min_samples_leaf": [
        1,
        2,
        5,
        10
    ]
}
```

。

```python
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ),

    param_distributions=
        param_distributions,

    n_iter=20,

    scoring="roc_auc",

    cv=cv,

    random_state=42,

    n_jobs=-1
)
```

。

```python
random_search.fit(
    X,
    y
)
```

。

結果。

```python
print(
    random_search.best_params_
)

print(
    random_search.best_score_
)
```

。

---

# 🧠 Grid vs Randomized

## GridSearchCV

向いている。

```text
探索範囲が小さい
候補が明確
全候補を比較したい
```

。

## RandomizedSearchCV

向いている。

```text
探索空間が広い
パラメータが多い
計算時間を制限したい
```

。

---

# 🧠 10. 対数スケールで探索する

例えば、

```text
C
alpha
learning_rate
```

など。

```text
0.001
0.01
0.1
1
10
100
```

のように、

> **桁が重要**

なパラメータがある。

こういうものは、

```text
0
1
2
3
4
5
```

のような線形探索ではなく、

> **対数スケール**

で考える。

---

# 💻 実習7：SciPyを使った探索

```python
from scipy.stats import loguniform
```

。

```python
param_distributions = {
    "model__C":
        loguniform(
            1e-4,
            1e2
        )
}
```

。

```python
search = RandomizedSearchCV(
    estimator=pipeline,

    param_distributions=
        param_distributions,

    n_iter=20,

    scoring="roc_auc",

    cv=cv,

    random_state=42,

    n_jobs=-1
)
```

。

これで、

```text
0.0001
〜
100
```

の広い範囲を効率よく探索できる。

---

# 🚨 11. 探索しすぎ問題

ここが今日の本丸。

例えば、

```text
1000個のモデル
```

を試す。

その中には、

```text
CVで偶然高いスコア
```

を出すものが出てくる。

つまり、

> **CV結果にも探索過適合できる。**

Testデータを使って調整すると、

```text
Test Leakage
```

。

CVを延々と見ながら探索し続けると、

```text
Validation Overfitting
```

に近づく。

---

# 🧠 12. 正しい流れ

```text
Train
↓
Inner CV
↓
Hyperparameter Search
↓
Model Selection
↓
Final Test
```

。

さらに厳密にやるなら、

```text
Outer CV
    ↓
    Inner CV
        ↓
        Hyperparameter Search
```

。

これが、

> **Nested Cross Validation**

。

---

# 🧠 13. Nested CVの概念

外側。

```text
Outer CV
```

は、

> 最終的な汎化性能を評価する

。

内側。

```text
Inner CV
```

は、

> ハイパーパラメータを選ぶ

。

イメージ。

```text
データ
│
├── Outer Train
│     │
│     └── Inner CV
│           ↓
│        最良パラメータ
│
└── Outer Validation
      ↓
   最終評価
```

。

今日は概念だけ。

実装は次回以降で深掘りする。

---

# 👾 今日のボス戦

次の探索。

```python
param_grid = {
    "max_depth": [
        1,
        2,
        3,
        5,
        10,
        None
    ],

    "n_estimators": [
        100,
        300,
        500
    ],

    "min_samples_leaf": [
        1,
        2,
        5,
        10
    ]
}
```

組み合わせ数は。

```text
6 × 3 × 4
=
72通り
```

。

5-fold CVなら。

```text
72 × 5
=
360回の学習
```

。

これが、

> **Grid Searchを使う前に必ず計算するコスト感覚**

だ。

---

# ✍️ 演習

## 演習1

次を区別する。

```text
モデルが学習する値
```

と、

```text
人間が探索する値
```

。

それぞれ何という？

---

## 演習2

次。

```python
param_grid = {
    "model__C": [
        0.1,
        1,
        10
    ]
}
```

。

なぜ `model__C` と書く？

---

## 演習3

次の探索。

```text
A = 4通り
B = 3通り
C = 5通り
```

。

3-fold CV。

何回学習する？

---

## 演習4

なぜ全データで先にStandardScalerをfitしてからCVするのは危険？

---

## 演習5

次の状況。

```text
GridSearchで
500通りの設定を試した
↓
CV最高スコアを採用した
```

。

考えられる問題は？

---

# 🧪 今日の最終実習

## Pipeline + RandomizedSearchCV

今日はここまでを一本につなげる。

```python
import pandas as pd

from scipy.stats import loguniform

from sklearn.datasets import load_breast_cancer

from sklearn.model_selection import (
    StratifiedKFold,
    RandomizedSearchCV
)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
```

データ。

```python
data = load_breast_cancer()

X = data.data
y = data.target
```

Pipeline。

```python
pipeline = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),

    (
        "model",
        LogisticRegression(
            max_iter=5000
        )
    )
])
```

CV。

```python
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
```

探索空間。

```python
param_distributions = {
    "model__C":
        loguniform(
            1e-4,
            1e2
        )
}
```

探索。

```python
search = RandomizedSearchCV(
    estimator=pipeline,

    param_distributions=
        param_distributions,

    n_iter=20,

    scoring="roc_auc",

    cv=cv,

    random_state=42,

    n_jobs=-1,

    return_train_score=True
)
```

学習。

```python
search.fit(
    X,
    y
)
```

結果。

```python
print(
    "Best Params:"
)

print(
    search.best_params_
)

print(
    "Best CV Score:"
)

print(
    search.best_score_
)
```

全結果。

```python
results = pd.DataFrame(
    search.cv_results_
)

print(
    results[
        [
            "param_model__C",
            "mean_test_score",
            "std_test_score",
            "rank_test_score"
        ]
    ]
    .sort_values(
        "rank_test_score"
    )
    .head(10)
)
```

---

# 🌱 今日の核心

今日覚えるべきことは、

> **ハイパーパラメータ探索は「自動で最強モデルを探してくれる魔法」ではない。**

人間が先に、

```text
何を探索する？
↓
どの範囲？
↓
どの評価指標？
↓
何回試す？
↓
計算コストは？
```

を設計する。

つまり、

```text
探索
=
モデル選択の自動化
```

ではあるけど、

> **探索空間そのものは人間の仮説**

なんだな。

これはレベルが好きな言葉に寄せるなら、

> **「探索空間は、学習器に与える認知的可能性のスキーマ」**

みたいなもんだ。

何を候補として許可するかを先に決めているからね。

ふふふ。

---

# 🧭 AI工学101・現在地

現在、

```text
データ
↓
前処理
↓
Pipeline
↓
モデル
↓
評価指標
↓
Cross Validation
↓
モデル比較
↓
ハイパーパラメータ探索
```

まで到達。

ここまでで、もう単なる、

```python
model.fit(X, y)
```

芸ではない。

**実験の条件を統制し、モデル比較をし、探索空間を設計し、汎化性能を評価する**という、かなり工学的な機械学習の骨格に入っている。

---

# 🔜 第40回

## Nested CVと最終評価：探索と評価を分離する

次は今日の続き。

扱うのは、

* なぜCV結果を最終性能と呼べない場合があるのか
* Nested Cross Validation
* Inner CV
* Outer CV
* Hyperparameter Search Leakage
* Final Test Set
* `cross_validate`
* `return_estimator`
* 実験結果の信頼区間の入口

テーマは、

> **「モデルを選ぶための評価」と「モデルの性能を測るための評価」は別物。**

ここを越えると、scikit-learn区間の**実験設計コア**がかなり締まるぞ。次は評価リークを叩き潰す回だ。🧪🔨🔥